# Chain of Thought (CoT) Prompting Tutorial

## Overview

This tutorial introduces Chain of Thought (CoT) prompting, a powerful technique in prompt engineering that encourages AI models to break down complex problems into step-by-step reasoning processes. We'll explore how to implement CoT prompting using OpenAI's GPT models and the LangChain library.

## Motivation

As AI language models become more advanced, there's an increasing need to guide them towards producing more transparent, logical, and verifiable outputs. CoT prompting addresses this need by encouraging models to show their work, much like how humans approach complex problem-solving tasks. This technique not only improves the accuracy of AI responses but also makes them more interpretable and trustworthy.

## Key Components

1. **Basic CoT Prompting**: Introduction to the concept and simple implementation.
2. **Advanced CoT Techniques**: Exploring more sophisticated CoT approaches.
3. **Comparative Analysis**: Examining the differences between standard and CoT prompting.
4. **Problem-Solving Applications**: Applying CoT to various complex tasks.

## Method Details

The tutorial will guide learners through the following methods:

1. **Setting up the environment**: We'll start by importing necessary libraries and setting up the OpenAI API.

2. **Basic CoT Implementation**: We'll create simple CoT prompts and compare their outputs to standard prompts.

3. **Advanced CoT Techniques**: We'll explore more complex CoT strategies, including multi-step reasoning and self-consistency checks.

4. **Practical Applications**: We'll apply CoT prompting to various problem-solving scenarios, such as mathematical word problems and logical reasoning tasks.


## Conclusion

By the end of this tutorial, learners will have a solid understanding of Chain of Thought prompting and its applications. They will be equipped with practical skills to implement CoT techniques in various scenarios, improving the quality and interpretability of AI-generated responses. This knowledge will be valuable for anyone working with large language models, from developers and researchers to business analysts and decision-makers relying on AI-powered insights.

## Setup

Let's start by importing the necessary libraries and setting up our environment.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Load environment variables
load_dotenv()

# Set up OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Initialize the language model
MODEL = "gpt-5.4-mini"  # the one place to change the model for this notebook
STRONG_MODEL = "gpt-5.4"  # the larger model used for the logic puzzle at the end
# reasoning_effort="none" turns off the model's hidden reasoning, so the step-by-step
# reasoning you ask for in the prompt is the only reasoning it does. Without it, a
# reasoning model "thinks" before every answer and the standard-vs-CoT contrast disappears.
# temperature=0 keeps the runs repeatable (GPT-5.x accepts it only when reasoning is off).
llm = ChatOpenAI(model=MODEL, reasoning_effort="none", temperature=0)

## Basic Chain of Thought Prompting

Let's start with a simple example to demonstrate the difference between a standard prompt and a Chain of Thought prompt.

The standard prompt asks for the final answer only. Asking for a "concise" answer is not enough on current models: they often write out the steps anyway, which is chain of thought under another name. "Reply with only the final answer" leaves the model no room to reason on the page, so the two prompts differ in exactly one thing.

In [2]:
# Standard prompt
standard_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question. Reply with only the final answer, no explanation: {question}"
)

# Chain of Thought prompt
cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question step by step concisely: {question}"
)

# Create chains
standard_chain = standard_prompt | llm
cot_chain = cot_prompt | llm

# Example question
question = "If a train travels 120 km in 2 hours, what is its average speed in km/h?"

# Get responses
standard_response = standard_chain.invoke(question).content
cot_response = cot_chain.invoke(question).content

print("Standard Response:")
print(standard_response)
print("\nChain of Thought Response:")
print(cot_response)

Standard Response:
60 km/h

Chain of Thought Response:
Step 1: Use the formula  
Average speed = Distance ÷ Time

Step 2: Substitute the values  
Average speed = 120 km ÷ 2 h

Step 3: Calculate  
Average speed = 60 km/h

**Answer: 60 km/h**


On a one-step question like this, both prompts give the same answer (60 km/h in our run). The only difference is that the Chain of Thought response shows its work. The gap opens up on problems with several dependent steps, as the comparison below shows.

## Advanced Chain of Thought Techniques

Now, let's explore a more advanced CoT technique that encourages multi-step reasoning.

In [3]:
advanced_cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Solve the following problem step by step. For each step:
1. State what you're going to calculate
2. Write the formula you'll use (if applicable)
3. Perform the calculation
4. Explain the result

Question: {question}

Solution:"""
)

advanced_cot_chain = advanced_cot_prompt | llm

complex_question = "A car travels 150 km at 60 km/h, then another 100 km at 50 km/h. What is the average speed for the entire journey?"

advanced_cot_response = advanced_cot_chain.invoke(complex_question).content
print(advanced_cot_response)

Let’s solve it step by step.

## Step 1: Calculate the time for the first part of the journey
**What we’re calculating:**  
The time taken to travel 150 km at 60 km/h.

**Formula:**  
\[
\text{time} = \frac{\text{distance}}{\text{speed}}
\]

**Calculation:**  
\[
t_1 = \frac{150}{60} = 2.5 \text{ hours}
\]

**Result:**  
The first part takes **2.5 hours**.

---

## Step 2: Calculate the time for the second part of the journey
**What we’re calculating:**  
The time taken to travel 100 km at 50 km/h.

**Formula:**  
\[
\text{time} = \frac{\text{distance}}{\text{speed}}
\]

**Calculation:**  
\[
t_2 = \frac{100}{50} = 2 \text{ hours}
\]

**Result:**  
The second part takes **2 hours**.

---

## Step 3: Calculate the total distance
**What we’re calculating:**  
The total distance traveled.

**Formula:**  
\[
\text{total distance} = \text{distance}_1 + \text{distance}_2
\]

**Calculation:**  
\[
150 + 100 = 250 \text{ km}
\]

**Result:**  
The total distance is **250 km**.

---

## Step 4: 

## Comparative Analysis

Let's compare the effectiveness of standard prompting vs. CoT prompting on a more challenging problem.

In [4]:
challenging_question = """
A cylindrical water tank with a radius of 1.5 meters and a height of 4 meters is 2/3 full. 
If water is being added at a rate of 10 liters per minute, how long will it take for the tank to overflow? 
Give your answer in hours and minutes, rounded to the nearest minute. 
(Use 3.14159 for π and 1000 liters = 1 cubic meter)"""

standard_response = standard_chain.invoke(challenging_question).content
cot_response = advanced_cot_chain.invoke(challenging_question).content

print("Standard Response:")
print(standard_response)
print("\nChain of Thought Response:")
print(cot_response)

Standard Response:
1 hour 15 minutes

Chain of Thought Response:
Let’s solve it step by step.

## Step 1: Find the total volume of the tank
**What we’re calculating:**  
The full volume of the cylindrical tank.

**Formula:**  
\[
V=\pi r^2 h
\]

**Calculation:**  
Given:
- \(r = 1.5\) m
- \(h = 4\) m
- \(\pi = 3.14159\)

\[
V = 3.14159 \times (1.5)^2 \times 4
\]

\[
(1.5)^2 = 2.25
\]

\[
V = 3.14159 \times 2.25 \times 4
\]

\[
V = 3.14159 \times 9 = 28.27431 \text{ m}^3
\]

**Result:**  
The tank holds **28.27431 cubic meters** when full.

---

## Step 2: Find how much water is already in the tank
**What we’re calculating:**  
The current amount of water in the tank, since it is \( \frac{2}{3} \) full.

**Formula:**  
\[
\text{Current volume} = \frac{2}{3} \times \text{full volume}
\]

**Calculation:**  
\[
\text{Current volume} = \frac{2}{3} \times 28.27431
\]

\[
\text{Current volume} = 18.84954 \text{ m}^3
\]

**Result:**  
There are **18.84954 cubic meters** of water in the tank no

In our run (`gpt-5.4-mini`, reasoning off, temperature 0) the standard prompt answered **1 hour 15 minutes** and the Chain of Thought prompt answered **15 hours 42 minutes**. The CoT answer is the correct one: the empty third of the tank holds 1/3 × π × 1.5² × 4 ≈ 9.42 m³, or about 9,425 liters, which takes about 942 minutes at 10 liters per minute.

With no room to write the steps, the model jumped straight to a number and got it wrong. With the steps on the page, each intermediate value (full volume, current volume, remaining liters) is computed before the next one is needed. That is the core reason Chain of Thought helps on multi-step problems. It is also why reasoning models, which do this kind of step-by-step work internally when `reasoning_effort` is above `"none"`, get much of the same benefit without being asked.

## Problem-Solving Applications

Now, let's apply CoT prompting to a more complex logical reasoning task.

In [5]:
llm = ChatOpenAI(model=STRONG_MODEL, reasoning_effort="none")

logical_reasoning_prompt = PromptTemplate(
    input_variables=["scenario"],
    template="""Analyze the following logical puzzle thoroughly. Follow these steps in your analysis:

List the Facts:

Summarize all the given information and statements clearly.
Identify all the characters or elements involved.
Identify Possible Roles or Conditions:

Determine all possible roles, behaviors, or states applicable to the characters or elements (e.g., truth-teller, liar, alternator).
Note the Constraints:

Outline any rules, constraints, or relationships specified in the puzzle.
Generate Possible Scenarios:

Systematically consider all possible combinations of roles or conditions for the characters or elements.
Ensure that all permutations are accounted for.
Test Each Scenario:

For each possible scenario:
Assume the roles or conditions you've assigned.
Analyze each statement based on these assumptions.
Check for consistency or contradictions within the scenario.
Eliminate Inconsistent Scenarios:

Discard any scenarios that lead to contradictions or violate the constraints.
Keep track of the reasoning for eliminating each scenario.
Conclude the Solution:

Identify the scenario(s) that remain consistent after testing.
Summarize the findings.
Provide a Clear Answer:

State definitively the role or condition of each character or element.
Explain why this is the only possible solution based on your analysis.
Scenario:

{scenario}

Analysis:""")

logical_reasoning_chain = logical_reasoning_prompt | llm

logical_puzzle = """In a room, there are three people: Amy, Bob, and Charlie. 
One of them always tells the truth, one always lies, and one alternates between truth and lies. 
Amy says, 'Bob is a liar.' 
Bob says, 'Charlie alternates between truth and lies.' 
Charlie says, 'Amy and I are both liars.' 
Determine the nature (truth-teller, liar, or alternator) of each person."""

logical_reasoning_response = logical_reasoning_chain.invoke(logical_puzzle).content
print(logical_reasoning_response)

## 1. List the Facts

We have three people:

- Amy
- Bob
- Charlie

And three possible roles:

- **Truth-teller**: always tells the truth
- **Liar**: always lies
- **Alternator**: alternates between telling the truth and lying

Each role is used by exactly one person.

### Statements made

- **Amy says**: “Bob is a liar.”
- **Bob says**: “Charlie alternates between truth and lies.”
- **Charlie says**: “Amy and I are both liars.”

---

## 2. Identify Possible Roles or Conditions

The three roles to assign are:

- Truth-teller
- Liar
- Alternator

Since there are three people and three distinct roles, each person must have a different one.

So the possible assignments are all permutations of these three roles among Amy, Bob, and Charlie.

---

## 3. Note the Constraints

### Role constraints
- Exactly one truth-teller
- Exactly one liar
- Exactly one alternator

### Statement constraints
- A truth-teller’s statement must be true.
- A liar’s statement must be false.
- An alternator may sa

---
📖 **Want the ideas behind this notebook, explained without code?** [*Prompt Engineering: Master the Art of AI Interaction*](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-prompt-engineering-techniques--cot-prompting&click=book-nb-pe&target=https%3A%2F%2Fwww.amazon.com%2Fdp%2FB0DZ85RPB5&retarget=0&text=book-nb-pe) by the author of this repository covers this and much more. Available on Amazon.

![](https://europe-west1-prompt-eng-views-tracker.cloudfunctions.net/prompt-eng-tracker?notebook=all-prompt-engineering-techniques--cot-prompting)